# Using Python to Verify AI output

In [45]:
# Import Packages
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns 

In [46]:
# Create timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print("Jeffrey Lieu")

Current Time:09_25_2026_0250PM
Jeffrey Lieu


In [47]:
# Read the dataset and store it as data2026
data2026=pd.read_csv('2026PartYear.csv',encoding='utf-8')

# use concatenate
# Read in the 2025 dataset as data2025 

# Read the dataset and store it as data2025
data2025=pd.read_csv('2025.csv',encoding='utf-8')
new=pd.concat([data2026,data2025])

In [48]:
for d in (data2025, data2026):
    d['Current Year Week Ending'] = pd.to_datetime(d['Current Year Week Ending'])

new = pd.concat([data2026, data2025], ignore_index=True)

VOL_COL = 'Total Bulk and Bags'   # pounds, per Geography/Type/week
ASP_COL = 'ASP Current Year'       # $/lb, per Geography/Type/week

# Both files run Period 1-32 for 2026; restrict 2025 to the same 32 weeks
# so the two years are comparing the same calendar stretch.
WEEKS = 32
d25 = data2025[data2025['Period'] <= WEEKS]
d26 = data2026[data2026['Period'] <= WEEKS]

REGIONS = ['Southeast', 'Midsouth', 'California', 'South Central',
           'Northeast', 'Great Lakes', 'Plains', 'West']

def wavg(df, geography, type_):
    """Volume-weighted average ASP and total volume for one Geography/Type."""
    sub = df[(df['Geography'] == geography) & (df['Type'] == type_)]
    vol = sub[VOL_COL].sum()
    asp = (sub[ASP_COL] * sub[VOL_COL]).sum() / vol
    return asp, vol


def pct_chg(new_val, old_val):
    return (new_val / old_val - 1) * 100

In [49]:
# National conventional & organic, weeks 1-32
conv_asp_25, conv_vol_25 = wavg(d25, 'Total U.S.', 'Conventional')
conv_asp_26, conv_vol_26 = wavg(d26, 'Total U.S.', 'Conventional')
org_asp_25, org_vol_25 = wavg(d25, 'Total U.S.', 'Organic')
org_asp_26, org_vol_26 = wavg(d26, 'Total U.S.', 'Organic')

org_share_25 = org_vol_25 / (org_vol_25 + conv_vol_25) * 100
org_share_26 = org_vol_26 / (org_vol_26 + conv_vol_26) * 100
    
combined_vol_25 = conv_vol_25 + org_vol_25
combined_vol_26 = conv_vol_26 + org_vol_26

print('NATIONAL, WEEKS 1-%d' % WEEKS)
print(f'  Conventional ASP:    ${conv_asp_25:.2f} -> ${conv_asp_26:.2f}  '
      f'({pct_chg(conv_asp_26, conv_asp_25):+.1f}%)')
print(f'  Conventional Volume: {conv_vol_25/1e9:.2f}B -> {conv_vol_26/1e9:.2f}B lb  '
      f'({pct_chg(conv_vol_26, conv_vol_25):+.1f}%)')
print(f'  Organic ASP:         ${org_asp_25:.2f} -> ${org_asp_26:.2f}  '
      f'({pct_chg(org_asp_26, org_asp_25):+.1f}%)')
print(f'  Organic Volume:      {org_vol_25/1e6:.1f}M -> {org_vol_26/1e6:.1f}M lb  '
      f'({pct_chg(org_vol_26, org_vol_25):+.1f}%)')
print(f'  Organic Share:       {org_share_25:.1f}% -> {org_share_26:.1f}%  '
      f'({org_share_26 - org_share_25:+.1f} pt)')
print(f'  Combined Volume:     {pct_chg(combined_vol_26, combined_vol_25):+.1f}%')
print(f'  Organic premium, 2026: {(org_asp_26/conv_asp_26 - 1)*100:.1f}%')

NATIONAL, WEEKS 1-32
  Conventional ASP:    $1.22 -> $0.98  (-19.4%)
  Conventional Volume: 1.76B -> 2.02B lb  (+14.5%)
  Organic ASP:         $1.74 -> $1.56  (-10.3%)
  Organic Volume:      105.9M -> 105.1M lb  (-0.7%)
  Organic Share:       5.7% -> 4.9%  (-0.7 pt)
  Combined Volume:     +13.6%
  Organic premium, 2026: 59.5%


In [50]:
# Weekly conventional ASP price cycle, Total U.S.
tus_conv = pd.concat([d25, d26])
tus_conv = tus_conv[(tus_conv['Geography'] == 'Total U.S.') &
                     (tus_conv['Type'] == 'Conventional')].sort_values('Current Year Week Ending')

peak_25 = tus_conv[tus_conv['Current Year Week Ending'].dt.year == 2025].pipe(
    lambda d: d.loc[d[ASP_COL].idxmax()])
trough_26 = tus_conv[tus_conv['Current Year Week Ending'].dt.year == 2026].pipe(
    lambda d: d.loc[d[ASP_COL].idxmin()])
latest = tus_conv.iloc[-1]

print('\nWEEKLY PRICE CYCLE (Total U.S., Conventional)')
print(f"  2025 peak:   ${peak_25[ASP_COL]:.2f}  (week of {peak_25['Current Year Week Ending'].date()})")
print(f"  2026 trough: ${trough_26[ASP_COL]:.2f}  (week of {trough_26['Current Year Week Ending'].date()})")
print(f"  Latest week: ${latest[ASP_COL]:.2f}  (week of {latest['Current Year Week Ending'].date()})")


WEEKLY PRICE CYCLE (Total U.S., Conventional)
  2025 peak:   $1.31  (week of 2025-04-20)
  2026 trough: $0.89  (week of 2026-02-08)
  Latest week: $1.06  (week of 2026-08-09)


In [51]:
# Regional conventional ASP & volume, weeks 1-32
print('\nREGIONAL, CONVENTIONAL, WEEKS 1-%d' % WEEKS)
print(f"  {'Region':16s} {'Vol YoY':>9s} {'ASP YoY':>9s} {'2026 ASP':>10s}")
for r in REGIONS:
    asp25, vol25 = wavg(d25, r, 'Conventional')
    asp26, vol26 = wavg(d26, r, 'Conventional')
    print(f"  {r:16s} {pct_chg(vol26, vol25):8.1f}% {pct_chg(asp26, asp25):8.1f}% "
          f"${asp26:9.2f}")


REGIONAL, CONVENTIONAL, WEEKS 1-32
  Region             Vol YoY   ASP YoY   2026 ASP
  Southeast            20.7%    -25.6% $     0.87
  Midsouth             17.1%    -18.2% $     0.96
  California           15.6%    -17.9% $     1.14
  South Central        14.6%    -21.6% $     0.83
  Northeast            14.4%    -15.7% $     0.98
  Great Lakes          14.2%    -17.9% $     0.97
  Plains               10.8%    -20.1% $     0.89
  West                  7.4%    -18.6% $     1.10


In [52]:
# Check: is Nov 2025-Apr 2026 actually the low-price window?
low_window = tus_conv[(tus_conv['Current Year Week Ending'] >= '2025-11-01') &
                       (tus_conv['Current Year Week Ending'] <= '2026-04-30')]
rest = tus_conv[~tus_conv.index.isin(low_window.index)]

print('\nPROMOTIONAL WINDOW CHECK')
print(f"  Avg ASP, Nov 2025-Apr 2026: ${low_window[ASP_COL].mean():.2f}")
print(f"  Avg ASP, rest of year:      ${rest[ASP_COL].mean():.2f}")


PROMOTIONAL WINDOW CHECK
  Avg ASP, Nov 2025-Apr 2026: $0.92
  Avg ASP, rest of year:      $1.16


In [53]:
# Check: March -> June 2026 price recovery claim ($0.92 -> $1.13)
march_2026 = tus_conv[(tus_conv['Current Year Week Ending'] >= '2026-03-01') &
                       (tus_conv['Current Year Week Ending'] <= '2026-03-07')]
june_2026 = tus_conv[(tus_conv['Current Year Week Ending'] >= '2026-06-24') &
                      (tus_conv['Current Year Week Ending'] <= '2026-06-30')]

print('PRICE RECOVERY CHECK')
print(f"  Early March 2026 ASP: ${march_2026[ASP_COL].iloc[0]:.2f}  "
      f"(week of {march_2026['Current Year Week Ending'].iloc[0].date()})")
print(f"  Late June 2026 ASP:   ${june_2026[ASP_COL].iloc[-1]:.2f}  "
      f"(week of {june_2026['Current Year Week Ending'].iloc[-1].date()})")

PRICE RECOVERY CHECK
  Early March 2026 ASP: $0.92  (week of 2026-03-01)
  Late June 2026 ASP:   $1.19  (week of 2026-06-28)


In [54]:
# Check: memo's "Data scope" bullet says 2026 Periods 1-8
print('\nDATA SCOPE CHECK')
print(f"  2026 file actual Period range: {data2026['Period'].min()} to {data2026['Period'].max()}")
print(f"  2026 file actual week range:   {data2026['Current Year Week Ending'].min().date()} "
      f"to {data2026['Current Year Week Ending'].max().date()}")
print("  Memo's 'Data scope' bullet claims: 2026 Periods 1-8, week ending 8/9/26")


DATA SCOPE CHECK
  2026 file actual Period range: 1 to 32
  2026 file actual week range:   2026-01-04 to 2026-08-09
  Memo's 'Data scope' bullet claims: 2026 Periods 1-8, week ending 8/9/26
